In [1]:
!huggingface-cli whoami


⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
Not logged in


In [2]:
import json
from pprint import pprint
import torch
import os
import ast
import pandas as pd
import concurrent.futures

# device = torch.device("cuda:1")
print("CUDA available:", torch.cuda.is_available())
print("Number of CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Number of CUDA devices: 2
Current CUDA device: 0
CUDA device name: NVIDIA RTX A5000


In [3]:
# read csv file

df = pd.read_csv('o4mini_annotation_final.csv')
df.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,t_result,t_corrected_version,t_category,t_notes,h_result,...,llm_errors,num_groups,agreement,text_length,llm_result_final,issues,llm_a1_agreement,llm_a2_agreement,a1_a2_agreement,llm_final_agreement
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",1,NaN,NaN,NaN,1,...,NaN,4.0,TRUE,110,"[\n {\n ""Chest pain"": [\n ""non stop c...",[],TRUE,TRUE,TRUE,TRUE
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...","""No temporal information found""",No temporal information found,1,NaN,NaN,NaN,1,...,NaN,NaN,TRUE,18,No temporal information found,[],TRUE,TRUE,TRUE,TRUE
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",Symptoms with no original mention should be gr...,NaN,0,...,NaN,2.0,FALSE,13,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","['llm formatting issue', 'incorrect grouping',...",FALSE,FALSE,TRUE,FALSE
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",1,NaN,NaN,NaN,0,...,NaN,2.0,0,260,"[\n {\n ""Vaccination site pain"": [\n ...",[],TRUE,0,0,TRUE
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",1,NaN,NaN,NaN,1,...,NaN,2.0,1,118,"[\n {\n ""Syncope"": [\n ""patient faint...",[],1,1,1,1


In [4]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 328.2+ KB


In [ ]:
# Gold timeline = human-corrected version when one exists, otherwise the accepted
# o4-mini result. A report belongs to MedTempo-T unless that gold timeline is
# "No temporal information found".
data['gold_timeline'] = data['final_corrected_version'].where(
    data['final_corrected_version'].notna(), data['llm_result_formatted'])
data['no_temporal_info'] = data['gold_timeline'].astype(str).str.lower().str.contains('no temporal information')

# # check for no temporal information content from final_corrected_version first and then llm_result_formatted if final_corrected_version is NaN in new column "no_temporal_info_final"
# data['no_temporal_info_final'] = data.apply(lambda row:
#     'no temporal information' in str(row['final_corrected_version']).lower() if not pd.isna(row['final_corrected_version']) else
#     'no temporal information' in str(row['llm_result_formatted']).lower(),
#     axis=1
# )
# print(data['no_temporal_info_final'].value_counts())

# print(data['no_temporal_info'].sum())
# print(data['no_temporal_info_final'].sum())

# # Find rows where they disagree
# diff = data[data['no_temporal_info'] != data['no_temporal_info_final']]
# print(diff[['final_corrected_version', 'llm_result_formatted', 'no_temporal_info', 'no_temporal_info_final']])


# get rows where no_temporal_info is False as new df
data_with_temporal_info = data[data['no_temporal_info'] == False]
data_with_temporal_info.info()
# we have to remove final_result value 2 from data_with_temporal_info
data_with_temporal_info = data_with_temporal_info[data_with_temporal_info['final_result'] != 2]
data_with_temporal_info.info()
# check final_result value_counts
data['final_result'].value_counts()

In [6]:
# find data with no temporal info
data_no_temporal_info = data[data['no_temporal_info'] == True]
data_no_temporal_info = data_no_temporal_info[data_no_temporal_info['final_result'] != 2]

data_no_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2191 entries, 1 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 2191 non-null   int64 
 1   symptom_text             2191 non-null   object
 2   symptom_list             2191 non-null   object
 3   llm_result               2191 non-null   object
 4   llm_result_formatted     2191 non-null   object
 5   final_corrected_version  238 non-null    object
 6   final_result             2191 non-null   int64 
 7   no_temporal_info         2191 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 139.1+ KB


In [7]:
# take 50 each from data_with_temporal_info and data_no_temporal_info
data_sampled_with_temporal_info = data_with_temporal_info
data_sampled_no_temporal_info = data_no_temporal_info
# concatenate the two dataframes
data_sampled = pd.concat([data_sampled_with_temporal_info, data_sampled_no_temporal_info])
data_sampled.info()
# save to csv
# data_sampled.to_csv('sample_40.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 5347 entries, 0 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 5347 non-null   int64 
 1   symptom_text             5347 non-null   object
 2   symptom_list             5347 non-null   object
 3   llm_result               5347 non-null   object
 4   llm_result_formatted     5347 non-null   object
 5   final_corrected_version  889 non-null    object
 6   final_result             5347 non-null   int64 
 7   no_temporal_info         5347 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 339.4+ KB


In [ ]:
# now lets take 40 random samples from data_with_temporal_info with random_state=2
sampled_data = data_sampled_with_temporal_info 
sampled_data.info()


In [9]:
# eg = data_with_temporal_info[['vaers_id','symptom_text', 'symptom_list', 'llm_result_formatted','final_corrected_version','final_result']]
# eg.to_csv('openai_gold_with_temporal.csv', index=False)

In [10]:
# assume temporal information exists for all
example = """
[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]
"""

input_template_mention = """
Ignore previous conversations.

Clinical Notes: {text}

Task:
    - Use the adverse effect list below:
     {suggest}
   - Reorder the list based on the sequence implied in the clinical notes.
   - If several adverse effects are mentioned **together without a clear temporal order**, group them into the same block (inside a single dictionary).
   - Do **not skip any terms** from the list — include all terms in the output.
   - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

Important Rules:
- Mention each symptom **only once** — at the first time it appears or becomes relevant.
- Do **NOT** treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
- If multiple symptoms are mentioned across multiple sentences **without clear timeline separation**, **group them together**.
- Only separate symptoms into different blocks when the clinical notes clearly describe a **sequence or timing** between them.

Finally, return the result **only** in valid JSON format —
using a **list of dictionaries** to represent temporal progression and grouping, as shown below:
{example}
"""

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# HuggingFace model (change in-code)
# =========================
MODEL_ID =  "meta-llama/Llama-3.1-8B-Instruct" # e.g., "google/medgemma-27b-text-it" or "meta-llama/Llama-3.1-8B-Instruct" or "unsloth/medgemma-27b-text-it-bnb-4bit","meta-llama/Llama-3.3-70B-Instruct"
USE_4BIT = False                               # True for 4-bit quantized loading (bitsandbytes)
MAX_NEW_TOKENS = 512

# Optional: if the repo is gated, set a HF token here (or login with huggingface-cli)
HF_TOKEN = None  # e.g., "hf_..."

# =========================
# CUDA / dtype config
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    # Prefer bf16 when supported; otherwise fp16
    DTYPE = torch.bfloat16 if (hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported()) else torch.float16
    print(f"Using CUDA: {torch.cuda.get_device_name(0)} | dtype={DTYPE}")
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DTYPE = torch.float32
    print("Using CPU | dtype=float32")

_hf_tokenizer = None
_hf_model = None

def reset_model():
    """Call this after changing MODEL_ID / USE_4BIT to reload without restarting the kernel."""
    global _hf_tokenizer, _hf_model
    _hf_tokenizer = None
    _hf_model = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _load_hf_once():
    global _hf_tokenizer, _hf_model
    if _hf_model is not None and _hf_tokenizer is not None:
        return

    quant_config = None
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=DTYPE,
        )

    _hf_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
        trust_remote_code=False,
        token=HF_TOKEN,
    )

    # Some repos don't set pad token; align it to EOS to avoid generate() warnings
    if _hf_tokenizer.pad_token_id is None and _hf_tokenizer.eos_token_id is not None:
        _hf_tokenizer.pad_token = _hf_tokenizer.eos_token

    if USE_4BIT:
        # Quantized: let Accelerate place weights on GPU/CPU automatically
        _hf_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="balanced",
            torch_dtype=DTYPE,
            quantization_config=quant_config,
            trust_remote_code=False,
            token=HF_TOKEN,
        )
    else:
        # Non-quantized: explicitly move to CUDA if available
        _hf_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="balanced",
            torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
            trust_remote_code=False,
            token=HF_TOKEN,
        ).to(DEVICE)

    _hf_model.eval()

def _model_input_device():
    # Works for both normal and device_map='auto' models
    try:
        return next(_hf_model.parameters()).device
    except StopIteration:
        return torch.device(DEVICE)


SYSTEM_PROMPT = (
    "You are a JSON generator. Output ONLY valid JSON. "
    "No markdown fences, no code, no explanations, no questions."
)

def _get_eos_ids(tokenizer):
    eos_ids = []
    if tokenizer.eos_token_id is not None:
        eos_ids.append(tokenizer.eos_token_id)

    # Common chat turn tokens across models
    for tok in ["<|eot_id|>", "<|end_of_turn|>", "<|endoftext|>", "</s>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        # convert_tokens_to_ids can return int; ignore unknowns / duplicates
        if isinstance(tid, int) and (tokenizer.unk_token_id is None or tid != tokenizer.unk_token_id):
            if tid not in eos_ids:
                eos_ids.append(tid)

    if len(eos_ids) == 0:
        return None
    return eos_ids[0] if len(eos_ids) == 1 else eos_ids

from  collections import Counter
def llm(prompt: str) -> str:
    """Drop-in replacement: same llm(prompt) API as before."""
    _load_hf_once()
    # print("Model class:", type(_hf_model))

    # # For accelerate-sharded models:
    # if hasattr(_hf_model, "hf_device_map"):
    #     c = Counter(_hf_model.hf_device_map.values())
    #     print("hf_device_map device counts:", c)
    #     # show a few entries
    #     for k in list(_hf_model.hf_device_map.keys())[:20]:
    #         print(k, "->", _hf_model.hf_device_map[k])
    # else:
    #     # Single-device model
    #     dev = next(_hf_model.parameters()).device
    #     print("Single device:", dev)

    # # Show PyTorch's view of current allocations per GPU (process-local)
    # for i in range(torch.cuda.device_count()):
    #     alloc = torch.cuda.memory_allocated(i) / 1024**3
    #     reserv = torch.cuda.memory_reserved(i) / 1024**3
    #     print(f"GPU{i}: allocated={alloc:.2f} GiB | reserved={reserv:.2f} GiB")


    EOS_IDS = _get_eos_ids(_hf_tokenizer)

    """
    Generates completion text for the user_prompt.
    Uses chat template when available (critical for instruct/chat models).
    """
    # Chat template path (best for Llama/Mistral/Gemma/MedGemma instruct variants)
    if hasattr(_hf_tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        rendered = _hf_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = _hf_tokenizer(rendered, return_tensors="pt")
    else:
        # Fallback (base models or older tokenizers)
        inputs = _hf_tokenizer(prompt, return_tensors="pt")

    inputs = {k: v.to(_hf_model.device) for k, v in inputs.items()}

    # Deterministic decoding for stability
    gen_kwargs = dict(
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=EOS_IDS,
        pad_token_id=(_hf_tokenizer.eos_token_id
                      if getattr(_hf_tokenizer, "eos_token_id", None) is not None
                      else (EOS_IDS[0] if isinstance(EOS_IDS, list) else EOS_IDS)),
    )

    with torch.no_grad():
        out = _hf_model.generate(**inputs, **gen_kwargs)

    # Decode ONLY newly generated tokens (prevents prompt echo)
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    text = _hf_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    del out, gen_ids
    torch.cuda.empty_cache()
    return text


def get_ordered_symptoms(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.append(list(sym_dict.keys()))
    return symptoms

def call_llm(row):
    prompt = input_template_mention.format(
        text=row['symptom_text'],
        suggest=row['symptom_list'],
        example=example
    )
    return llm(prompt)


Using CUDA: NVIDIA RTX A5000 | dtype=torch.bfloat16


In [ ]:

batch_size = 500  # this is a *checkpoint* chunk size, not a GPU micro-batch size

for start in range(500, len(sampled_data), batch_size):
    end = min(start + batch_size, len(sampled_data))
    print(f"Processing rows {start} to {end}...")
    batch = sampled_data.iloc[start:end]

    out = []
    for _, row in batch.iterrows():
        out.append(call_llm(row))

    # Assign results back to the correct rows
    sampled_data.loc[batch.index, "hf_result"] = out

    # Optional: free unused cached memory between chunks
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Save checkpoint
    sampled_data.to_csv(f"medgemma_with_temporal_baseline_part_{start}.csv", index=False)



In [13]:
import re
def parse_json_best_effort(text: str):
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (text.find("{"), text.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(text) - 1, start, -1):
            if text[end] in "}]":
                chunk = text[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None

In [14]:
# for row in sampled_data.iterrows():
#     res = row[1]['hf_result']
#     print("----------")
#     print(res)
#     print("----------")
#     print(res[-800:])
#     break
    

In [15]:
res_lst = []
for row in sampled_data.iterrows():
    res = parse_json_best_effort(row[1]['hf_result'])
    # print(res)
    
    res_lst.append(res)

# store res_lst in reslts_df
sampled_data['final_result_cleaned'] = res_lst
sampled_data.head()



,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,final_corrected_version,final_result,no_temporal_info,hf_result,final_result_cleaned
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",NaN,1,False,nan,None
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,False,nan,None
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",NaN,1,False,nan,None
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",NaN,1,False,nan,None
5,940509,"about 4 hrs after vaccination ee had ""tunnel ...","['Dizziness', 'Faeces soft', 'Fatigue', 'Flatu...","[\n {\n ""Tunnel vision"": [""tunnel vision""]...","[\n {\n ""Tunnel vision"": [\n ""tunnel ...",NaN,1,False,nan,None


In [16]:
sampled_data.to_csv(f"meggemma_with_temporal_baseline_final.csv", index=False)  # Save checkpoint


### investigating llama output

In [ ]:
# read in medgemma_with_temporal_baseline_part_3000.csv and medgemma_with_temporal_baseline_part_0.csv and concatenate them into one df
df1 = pd.read_csv('medgemma_with_temporal_baseline_part_3000.csv')
df2 = pd.read_csv('medgemma_with_temporal_baseline_part_0.csv')
gate_df = pd.concat([df2, df1], ignore_index=True)
# print(gate_df.head(5))
# remove duplicates by nan in final_result_cleaned
gate_df = gate_df.dropna(subset=['hf_result'])
gate_df.info()

In [28]:
res_lst = []
for row in gate_df.iterrows():
    res = parse_json_best_effort(row[1]['hf_result'])
    # print(res)
    
    res_lst.append(res)

# store res_lst in reslts_df
gate_df['final_result_cleaned'] = res_lst
gate_df.head(10)



,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,final_corrected_version,final_result,no_temporal_info,hf_result,final_result_cleaned
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",NaN,1,False,"```json\n[\n {\n ""Chest discomfort"": [""non...",[{'Chest discomfort': ['non stop chest pain th...
1,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,False,"```json\n[\n {\n ""Immediate post-injection...",[{'Immediate post-injection reaction': ['Muscl...
2,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",NaN,1,False,"```json\n[\n {\n ""Vaccination site pain"": ...",[{'Vaccination site pain': ['injection site pa...
3,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",NaN,1,False,"```json\n[\n {""Syncope"": [""patient fainted""]}...","[{'Syncope': ['patient fainted']}, {'Fall': ['..."
4,940509,"about 4 hrs after vaccination ee had ""tunnel ...","['Dizziness', 'Faeces soft', 'Fatigue', 'Flatu...","[\n {\n ""Tunnel vision"": [""tunnel vision""]...","[\n {\n ""Tunnel vision"": [\n ""tunnel ...",NaN,1,False,"```json\n[\n {\n ""Tunnel vision"": [""tunnel...","[{'Tunnel vision': ['tunnel vision'], 'Dizzine..."
5,1797468,also had wheezing and would get short of brea...,"['Asthenia', 'Dyspnoea', 'Nasal congestion', '...","[\n {\n ""Asthenia"": [""not much energy""],\n...","[\n {\n ""Asthenia"": [\n ""not much ene...",NaN,1,False,"```json\n[\n {\n ""Nasal congestion"": [""con...",[{'Nasal congestion': ['congestion at time of ...
6,1774261,"Stomach ache, cramps sometimes, after second v...","['Abdominal pain upper', 'Cough', 'Oropharynge...","[\n {\n ""Cough"": [""cough""]\n },\n {\n ...","[\n {\n ""Cough"": [\n ""cough""\n ]\n...",NaN,1,False,"```json\n[\n {\n ""Abdominal pain upper"": [...","[{'Abdominal pain upper': ['stomach ache, cram..."
7,2107775,loose bowels/diarrhea; bloated and gassy; bloa...,"['Abdominal distension', 'Diarrhoea', 'Flatule...","[\n {\n ""Pain in extremity"": [\n ""sor...","[\n {\n ""Pain in extremity"": [\n ""sor...",NaN,1,False,"```json\n[\n {\n ""Abdominal distension"": [...",[{'Abdominal distension': ['bloated and gassy'...
8,946352,Head felt heavy/full/tingly starting about 6 p...,"['Arthralgia', 'Back pain', 'Chest pain', 'Fat...","[\n {\n ""Head discomfort"": [""Head felt hea...","[\n {\n ""Head discomfort"": [\n ""Head ...",NaN,1,False,"```json\n[\n {\n ""Head discomfort"": [""Head...",[{'Head discomfort': ['Head felt heavy/full/ti...
9,1263734,Client reports receiving Pfizer COVID 19 vacci...,"['Headache', 'Injection site pain', 'Injection...","[\n {\n ""Injection site pain"": [""sharp pai...","[\n {\n ""Injection site pain"": [\n ""s...",NaN,1,False,"```json\n[\n {\n ""Injection site pain"": [""...",[{'Injection site pain': ['sharp pain at vacci...


In [29]:
gate_df.to_csv(f"medgemma_with_temporal_baseline_final.csv", index=False)  # Save checkpoint

In [31]:
gate_df.head(10)

,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,final_corrected_version,final_result,no_temporal_info,hf_result,final_result_cleaned
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",NaN,1,False,"```json\n[\n {\n ""Chest discomfort"": [""non...",[{'Chest discomfort': ['non stop chest pain th...
1,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,False,"```json\n[\n {\n ""Immediate post-injection...",[{'Immediate post-injection reaction': ['Muscl...
2,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",NaN,1,False,"```json\n[\n {\n ""Vaccination site pain"": ...",[{'Vaccination site pain': ['injection site pa...
3,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",NaN,1,False,"```json\n[\n {""Syncope"": [""patient fainted""]}...","[{'Syncope': ['patient fainted']}, {'Fall': ['..."
4,940509,"about 4 hrs after vaccination ee had ""tunnel ...","['Dizziness', 'Faeces soft', 'Fatigue', 'Flatu...","[\n {\n ""Tunnel vision"": [""tunnel vision""]...","[\n {\n ""Tunnel vision"": [\n ""tunnel ...",NaN,1,False,"```json\n[\n {\n ""Tunnel vision"": [""tunnel...","[{'Tunnel vision': ['tunnel vision'], 'Dizzine..."
5,1797468,also had wheezing and would get short of brea...,"['Asthenia', 'Dyspnoea', 'Nasal congestion', '...","[\n {\n ""Asthenia"": [""not much energy""],\n...","[\n {\n ""Asthenia"": [\n ""not much ene...",NaN,1,False,"```json\n[\n {\n ""Nasal congestion"": [""con...",[{'Nasal congestion': ['congestion at time of ...
6,1774261,"Stomach ache, cramps sometimes, after second v...","['Abdominal pain upper', 'Cough', 'Oropharynge...","[\n {\n ""Cough"": [""cough""]\n },\n {\n ...","[\n {\n ""Cough"": [\n ""cough""\n ]\n...",NaN,1,False,"```json\n[\n {\n ""Abdominal pain upper"": [...","[{'Abdominal pain upper': ['stomach ache, cram..."
7,2107775,loose bowels/diarrhea; bloated and gassy; bloa...,"['Abdominal distension', 'Diarrhoea', 'Flatule...","[\n {\n ""Pain in extremity"": [\n ""sor...","[\n {\n ""Pain in extremity"": [\n ""sor...",NaN,1,False,"```json\n[\n {\n ""Abdominal distension"": [...",[{'Abdominal distension': ['bloated and gassy'...
8,946352,Head felt heavy/full/tingly starting about 6 p...,"['Arthralgia', 'Back pain', 'Chest pain', 'Fat...","[\n {\n ""Head discomfort"": [""Head felt hea...","[\n {\n ""Head discomfort"": [\n ""Head ...",NaN,1,False,"```json\n[\n {\n ""Head discomfort"": [""Head...",[{'Head discomfort': ['Head felt heavy/full/ti...
9,1263734,Client reports receiving Pfizer COVID 19 vacci...,"['Headache', 'Injection site pain', 'Injection...","[\n {\n ""Injection site pain"": [""sharp pai...","[\n {\n ""Injection site pain"": [\n ""s...",NaN,1,False,"```json\n[\n {\n ""Injection site pain"": [""...",[{'Injection site pain': ['sharp pain at vacci...


In [ ]:
import pandas as pd
import json 
df_check = pd.read_csv('medgemma_27b_baseline_with_temporal_sample_67_final_corrected.csv')
df_check.head()

,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,final_corrected_version,final_result,no_temporal_info,hf_result,final_result_cleaned
0,1286220,After 18 hours I ( patient) had normal chill/b...,"['Chills', 'Pain', 'Rash', 'Rash pruritic']","[\n {\n ""Chills"": [""normal chill""],\n ""...","[\n {\n ""Chills"": [\n ""normal chill""\...",NaN,1,False,"```json\n[\n {""Chills"": [""normal chill""], ""Pa...","[{'Chills': ['normal chill'], 'Pain': ['body p..."
1,1215031,About 4 days after I started having abdomen pa...,"['Abdominal pain', 'Contusion', 'Dyspnoea', 'H...","[\n {\n ""Abdominal pain"": [""abdomen pain, ...","[\n {\n ""Abdominal pain"": [\n ""abdome...",NaN,1,False,"```json\n[\n {\n ""Abdominal pain"": [""abdom...","[{'Abdominal pain': ['abdomen pain'], 'Vaginal..."
2,1214024,Upon receiving shot pain shot down arm to left...,"['Abdominal pain', 'Arthralgia', 'Chills', 'Di...","[\n {\n ""Immediate post-injection reaction...","[\n {\n ""Immediate post-injection reaction...",NaN,1,False,"```json\n[\n {\n ""Immediate post-injection...",[{'Immediate post-injection reaction': ['shot ...
3,1403315,"Intense chills about 10 hours after shot, lead...","['Chills', 'Headache', 'Pain in extremity', 'P...","[\n {\n ""Chills"": [\n ""Intense chills...","[\n {\n ""Chills"": [\n ""Intense chills...",NaN,1,False,"```json\n[\n {""Chills"": [""Intense chills abou...",[{'Chills': ['Intense chills about 10 hours af...
4,2501633,Narrative: 80 yo male with hx of diverticulosi...,"['Bundle branch block right', 'Catheterisation...","[\n {\n ""Chest pain"": [""chest pain started...","[\n {\n ""Chest pain"": [\n ""chest pain...",NaN,1,False,"```json\n[\n {""Chest pain"": [""chest pain""], ""...","[{'Chest pain': ['chest pain'], 'Pain in extre..."


In [ ]:
# print out both hf_result and final_result_cleaned line by line to check exact output and compare
for index, row in df_check.sample(5).iterrows():
    print(f"Row {index}:")
    print("HF Result:")
    print(row['hf_result'])
    print("Final Result Cleaned:")
    print(row['final_result_cleaned'])
    print("ground truth:")
    print(row['llm_result_formatted'])
    print("----------")
    


Row 176:
HF Result:
```json
[
  {"Aura": ["Visual aura 3 hours later"]},
  {"Headache": ["headache for 2 days"]},
  {
    "Nausea": ["sudden onset nausea"],
    "Vomiting": ["vomiting"],
    "Emotional distress": ["emotional upset"],
    "Somnolence": ["sleepiness"]
  },
  {"Miosis": ["pupils to me seemed very constricted on that day"]},
  {"Musculoskeletal stiffness": ["stiffness of neck for several days"]},
  {"Facial pain": ["right sided head/ face pain for 5 days"]},
  {"Disturbance in attention": ["difficulty concentrating/ reading"]}
]
```
Final Result Cleaned:
[{'Aura': ['Visual aura 3 hours later']}, {'Headache': ['headache for 2 days']}, {'Nausea': ['sudden onset nausea'], 'Vomiting': ['vomiting'], 'Emotional distress': ['emotional upset'], 'Somnolence': ['sleepiness']}, {'Miosis': ['pupils to me seemed very constricted on that day']}, {'Musculoskeletal stiffness': ['stiffness of neck for several days']}, {'Facial pain': ['right sided head/ face pain for 5 days']}, {'Disturban

In [ ]:
# eg['ordered_symptoms'] = eg['llm_result_cleaned'].apply(get_ordered_symptoms)

In [ ]:
# eg['ordered_symptoms_flatten'] = eg['llm_result_cleaned'].apply(get_ordered_symptoms_flatten)

In [ ]:
# eg[eg['is_symptom_list_correct'] == False]

In [ ]:
# sampled_data['is_symptom_list_correct'].value_counts()

In [ ]:
# sampled_data['llm_result'][sampled_data['is_symptom_list_correct'] == False].value_counts()

In [ ]:
# def check_length(row):
#     ordered_symptoms_list = len(row['ordered_symptoms_flatten'])
#     symptom_list = len(ast.literal_eval(row['symptom_list']))
#     return ordered_symptoms_list == symptom_list

# sampled_data['is_same_length'] = sampled_data.apply(check_length, axis=1)

In [ ]:
# sampled_data.to_excel('data/sampled_annotated_data.xlsx', index=False)